In [1]:
%%capture
%pip install cupy-cuda12x
%pip install spacy[cuda-autodetect]
%pip install pandas
%pip install ekphrasis


In [2]:
FND_ROOT = "/workspace/rumor-detection"
PLURALISMO_ROOT = f"{FND_ROOT}/datasets/datasets-fnd-pluralismo/OnlyRepliesTree"

In [3]:
import json
import os

import numpy as np
import pandas as pd
import spacy
from ekphrasis.classes.preprocessor import TextPreProcessor
from ekphrasis.classes.tokenizer import SocialTokenizer
from ekphrasis.dicts.emoticons import emoticons


In [4]:
# spacy.require_gpu()

In [5]:
text_processor = TextPreProcessor(
    # terms that will be normalized
    normalize=[
        "url",
        "email",
        "percent",
        "money",
        "phone",
        "user",
        "time",
        "date",
        "number",
    ],
    # terms that will be annotated
    # annotate={"hashtag", "allcaps", "elongated", "repeated", 'emphasis', 'censored'},
    fix_html=True,  # fix HTML tokens
    # corpus from which the word statistics are going to be used
    # for word segmentation
    segmenter="twitter",
    # corpus from which the word statistics are going to be used
    # for spell correction
    # corrector="twitter",
    unpack_hashtags=True,  # perform word segmentation on hashtags
    # unpack_contractions=True,  # Unpack contractions (can't -> can not)
    spell_correct_elong=False,  # spell correction for elongated words
    # select a tokenizer. You can use SocialTokenizer, or pass your own
    # the tokenizer, should take as input a string and return a list of tokens
    tokenizer=SocialTokenizer(lowercase=True).tokenize,
    # list of dictionaries, for replacing tokens extracted from the text,
    # with other expressions. You can pass more than one dictionaries.
    dicts=[emoticons],
)

/home/ismael/.local/lib/python3.11/site-packages/ekphrasis/classes/tokenizer.py:225: FutureWarning: Possible nested set at position 2190
  self.tok = re.compile(r"({})".format("|".join(pipeline)))


Reading twitter - 1grams ...
Reading twitter - 2grams ...
Reading english - 1grams ...


/home/ismael/.local/lib/python3.11/site-packages/ekphrasis/classes/exmanager.py:14: FutureWarning: Possible nested set at position 42
  regexes = {k.lower(): re.compile(self.expressions[k]) for k, v in


In [6]:
def loadAllPosts(PLURALISMO_ROOT):
    """
    Carga todos los posts en formato json desde el directorio ./post.

    Retorna:

    - all_posts, diccionario indexado por tweet id
    - labeled_posts, diccionaro con tweets etiquetados
    - number_of_tweets
    """

    def parseTwitterTree(tree_file):
        tree_data = list()
        for line in tree_file:
            f_first_part, _ = line.split("->")
            _, first_part = f_first_part.split(":")
            first_part = first_part.strip()
            first_part = first_part.replace("'", '"')
            tree_data.append(json.loads(first_part))
        return tree_data

    ### Obtener diccionario con todos los posts
    all_posts = {}
    labels = {}

    subtrees = ["false", "true"]
    subtrees_count = {"false": 0, "true": 0, "imprecise": 0}
    print("all_posts before:")
    print(len(all_posts))
    print(all_posts)

    def retrieve_replies(tweet_info, accumulator):
        if "replies" not in tweet_info or len(tweet_info["replies"]) == 0:
            return accumulator
        else:
            new_accumulator = []
            for tweet_reply in tweet_info["replies"]:
                # reply_id = tweet_reply["id"]
                new_accumulator += retrieve_replies(tweet_reply, [tweet_reply])
            return accumulator + new_accumulator

    for subtree_label in subtrees:
        subtree_file_list = os.listdir(os.path.join(PLURALISMO_ROOT, subtree_label))
        print(
            f"Working with folder {subtree_label} with {len(subtree_file_list)} files"
        )
        for file in subtree_file_list:
            if file.endswith(".json") and not file.endswith("_minf.json"):
                print(f"Working with file {file}")
                try:
                    with open(
                        os.path.join(PLURALISMO_ROOT, subtree_label, file), "r"
                    ) as f:
                        tweet_id = file.split("_")[1].split(".")[0]
                        tweet_dic = json.load(f)
                        for tweet_info in tweet_dic["data"]:
                            if "conversation_id" in tweet_info:
                                if tweet_info["conversation_id"] == str(tweet_id):
                                    all_posts[tweet_id] = tweet_info

                        assert all_posts[tweet_id] is not None

                        nested_replies = retrieve_replies(all_posts[tweet_id], [])
                        print(f"Got {len(nested_replies)} nested replies")
                        for reply in nested_replies:
                            all_posts[reply["id"]] = reply

                        subtrees_count[subtree_label] += 1
                        labels[tweet_id] = subtree_label

                except Exception as exc:
                    print("EXC:")
                    print(exc)

    print("all_posts after:")
    print(len(all_posts))
    print("Tweets etiquetados      : ", len(labels), " ", subtrees_count)

    seqs_lens = []
    labeled_posts = {}
    number_of_tweets = 0
    number_of_retweets = 0
    number_of_invalid_tweets = 0
    no_in_data = 0
    opened_files = 0

    for idx, (tweet_id, subtree_label) in enumerate(labels.items()):
        try:
            if tweet_id in all_posts:
                tree_path = os.path.join(
                    PLURALISMO_ROOT, subtree_label, f"treefile_{tweet_id}_minf.json"
                )
                with open(tree_path) as tree_file:
                    opened_files += 1
                    print(f"File {tree_path} opened correctly")

                    tree_data = parseTwitterTree(tree_file)

                    # ### Remover retweets
                    first = tree_data[0]

                    without_rt = list(filter(lambda t: t[0] != tweet_id, tree_data[1:]))
                    number_of_retweets = number_of_retweets + (
                        len(tree_data[1:]) - len(without_rt)
                    )
                    only_valid = list(filter(lambda t: t[0] in all_posts, without_rt))
                    number_of_invalid_tweets = number_of_invalid_tweets + (
                        len(without_rt) - len(only_valid)
                    )
                    seqs_lens.append(len(only_valid))

                    labeled_posts[tweet_id] = (labels[tweet_id], [first] + only_valid)
                    number_of_tweets = number_of_tweets + 1
            else:
                no_in_data = no_in_data + 1

        except Exception as e:
            print(e)

    assert opened_files == len(labels)

    print(
        "no_in_data              : ", no_in_data
    )  ## están etiquetados, pero no en los post
    print("number_of_tweets        : ", number_of_tweets)
    print("all_posts               : ", len(all_posts))
    print("number_of_retweets      : ", number_of_retweets)  ## En árbol de propagación
    print(
        "number_of_invalid_tweets: ", number_of_invalid_tweets
    )  ## En árbol de propagación

    # La red neuronal necesita un tamaño fijo para la secuencia (datos de entrada)
    # ¿Que largo de secuencia utilizar?
    counts = np.bincount(seqs_lens)  ## seqs_len sólo de los 753
    mode_seq_len = np.argmax(counts)
    mean_seq_len = int(np.mean(seqs_lens))
    min__seq_len = min(seqs_lens)
    max__seq_len = max(seqs_lens)

    print("len(seqs_lens)   : ", len(seqs_lens))
    print("min__seq_len: ", min__seq_len)
    print("max__seq_len: ", max__seq_len)
    print("mean_seq_len: ", mean_seq_len)
    print("mode_seq_len: ", mode_seq_len)

    tree_max_num_seq = mean_seq_len

    return (all_posts, labeled_posts, number_of_tweets, tree_max_num_seq, seqs_lens)

In [7]:
(all_posts, labeled_posts, number_of_tweets, tree_max_num_seq, seqs_lens) = (
    loadAllPosts(PLURALISMO_ROOT)
)

all_posts before:
0
{}
Working with folder false with 210 files
Working with file treefile_1299150874774831106.json
Got 27 nested replies
Working with file treefile_1430686841540390912.json
Got 24 nested replies
Working with file treefile_1435396460804284418.json
Got 44 nested replies
Working with file treefile_1419856439443607558.json
Got 12 nested replies
Working with file treefile_1213539607612215301.json
Got 915 nested replies
Working with file treefile_1394331250333204483.json
EXC:
'1394331250333204483'
Working with file treefile_1251034040746082306.json
Got 14 nested replies
Working with file treefile_1201319587842801664.json
Got 107 nested replies
Working with file treefile_1419760933656739844.json
Got 1098 nested replies
Working with file treefile_1435974369281560580.json
Got 24 nested replies
Working with file treefile_1427082050998181894.json
Got 156 nested replies
Working with file treefile_1337399183708483585.json
Got 156 nested replies
Working with file treefile_1197656125

In [8]:
len(labeled_posts)

214

In [9]:
d = {i: k for i, (k, _) in enumerate(labeled_posts.items())}

In [10]:
regular_seqs_lens = np.array(list(filter(lambda x: x < 320, seqs_lens)))
tree_max_num_seq = int(np.floor(np.mean(regular_seqs_lens)))
tree_max_num_seq

88

In [11]:
labeled_posts["1299150874774831106"]

('false',
 [['1299150874774831106', '0.0'],
  ['1299156214069317636', '1273.0'],
  ['1299156316770963458', '1297.0'],
  ['1299158535805304832', '1826.0'],
  ['1299159444501213184', '2043.0'],
  ['1299163257941487616', '2952.0'],
  ['1299165714604064769', '3538.0'],
  ['1299167915917422593', '4063.0'],
  ['1299170321149329408', '4636.0'],
  ['1299172254580957185', '5097.0'],
  ['1299176131145469952', '6021.0'],
  ['1299177417085509632', '6328.0'],
  ['1299177774389907456', '6413.0'],
  ['1299177779318190080', '6414.0'],
  ['1299177988769165313', '6464.0'],
  ['1299182343505182720', '7502.0'],
  ['1299185107551621120', '8161.0'],
  ['1299192707538190336', '9973.0'],
  ['1299263225922555910', '26786.0'],
  ['1299264335810834432', '27051.0'],
  ['1299287174219739137', '32496.0'],
  ['1299321743216119808', '40738.0'],
  ['1299341645020909568', '45483.0'],
  ['1299347186342395905', '46804.0'],
  ['1299354801143193606', '48620.0'],
  ['1299365560648687616', '51185.0'],
  ['1299366508884303872

In [12]:
len(all_posts)

101045

In [13]:
all_posts["1299150874774831106"]

{'author_id': '34992710',
 'conversation_id': '1299150874774831106',
 'created_at': '2020-08-28T01:04:40.000Z',
 'delta_time': 0.0,
 'id': '1299150874774831106',
 'lang': 'es',
 'replies': [{'author_id': '2334166898',
   'conversation_id': '1299150874774831106',
   'created_at': '2020-08-28T18:29:16.000Z',
   'delta_time': 62676.0,
   'id': '1299413759233003520',
   'in_reply_to_user_id': '34992710',
   'lang': 'es',
   'replies': [],
   'source': 'Twitter for Android',
   'text': '@hernan_sr Al final y no era q el retiró del 10% nos iba a transformar en un país bananero... Porq resulta q llevabamos un buen rato ya siéndolo 🤦🤦🤦 #CamionerosverguenzaNacional'},
  {'author_id': '1292299328145559552',
   'conversation_id': '1299150874774831106',
   'created_at': '2020-08-28T15:21:31.000Z',
   'delta_time': 51411.0,
   'id': '1299366508884303872',
   'in_reply_to_user_id': '34992710',
   'lang': 'es',
   'replies': [],
   'source': 'Twitter for Android',
   'text': '@hernan_sr Les queda poc

In [14]:
big_dataframe_dict = {}
for tweet_id, (label, tree_list) in labeled_posts.items():
    print(f"Processing tweet_id: {tweet_id} with label: {label}")

    big_dataframe_dict[tweet_id] = {
        "index": 0,
        "is_root": True,
        "label": label,
        "tree_id": tweet_id,
        "text": all_posts[tweet_id]["text"],
    }

    for idx, reply in enumerate(tree_list[1:], start=1):
        if reply[0] in all_posts:
            big_dataframe_dict[reply[0]] = {
                "index": idx,
                "is_root": False,
                "label": "n/a",
                "tree_id": tweet_id,
                "text": all_posts[reply[0]]["text"],
            }
        else:
            print(f"Reply {reply[0]} not found in all_posts")

Processing tweet_id: 1299150874774831106 with label: false
Processing tweet_id: 1430686841540390912 with label: false
Processing tweet_id: 1435396460804284418 with label: false
Processing tweet_id: 1419856439443607558 with label: false
Processing tweet_id: 1213539607612215301 with label: false
Processing tweet_id: 1251034040746082306 with label: false
Processing tweet_id: 1201319587842801664 with label: false
Processing tweet_id: 1419760933656739844 with label: false
Processing tweet_id: 1435974369281560580 with label: false
Processing tweet_id: 1427082050998181894 with label: false
Processing tweet_id: 1337399183708483585 with label: false
Processing tweet_id: 1358146903742025730 with label: false
Processing tweet_id: 1319766841003331589 with label: false
Processing tweet_id: 1337733374736289795 with label: false
Processing tweet_id: 1270160451452768257 with label: false
Processing tweet_id: 1296591514576068609 with label: false
Processing tweet_id: 1273823342500425728 with label: fal

In [15]:
len(big_dataframe_dict)

101045

In [16]:
big_df = pd.DataFrame.from_dict(big_dataframe_dict, orient="index")

In [17]:
big_df

,index,is_root,label,tree_id,text
1299150874774831106,0,True,false,1299150874774831106,Esto es el colmo:\nVíctor Manoli dueño de empr...
1299156214069317636,1,False,n/a,1299150874774831106,@hernan_sr @Eneatipo7 Uy k raro 🙄
1299156316770963458,2,False,n/a,1299150874774831106,@hernan_sr No tienen verguenza. Que asco me da...
1299158535805304832,3,False,n/a,1299150874774831106,"@hernan_sr Anciano decrépito ,cuanto te que de..."
1299159444501213184,4,False,n/a,1299150874774831106,@hernan_sr Está la pura cagada en el gobierno ...
...,...,...,...,...,...
1267349731489038338,915,False,n/a,1266030489250541568,@T13 Y que.menos con la terrible falta de conc...
1267350520420859904,916,False,n/a,1266030489250541568,@SofiaCr86963636 @T13 Ja ja entonces felicitem...
1267445948923416576,917,False,n/a,1266030489250541568,@T13 Grande Mañalich y Piñera....y a la gente ...
1267845244734902273,918,False,n/a,1266030489250541568,@T13 Entoces nosotros inventamos la enfermedad...


!python -m spacy download es_core_news_lg

In [19]:
nlp = spacy.load("es_core_news_lg")
# nlp.disable_pipes("parser")

In [20]:
def procesar_texto(texto):
    doc = nlp(texto)
    n_palabras = len([token for token in doc if not token.is_punct])
    n_oraciones = len(list(doc.sents))
    prom_oracion = n_palabras / n_oraciones if n_oraciones > 0 else 0
    return {
        "n_palabras": n_palabras,
        "n_oraciones": n_oraciones,
        "prom_oracion": prom_oracion,
    }

In [21]:
big_df_metrics = big_df["text"].apply(procesar_texto).apply(pd.Series)

In [22]:
big_df = pd.concat([big_df, big_df_metrics], axis=1)

In [30]:
big_df.groupby(["tree_id"]).mean(["n_palabras"]).sort_values(
    by="n_palabras", ascending=False
)

,index,is_root,n_palabras,n_oraciones,prom_oracion
tree_id,,,,,
1299382969480679426,0.5,0.500000,39.500000,3.000000,13.000000
1215464348686192640,14.5,0.033333,37.266667,2.366667,19.815556
1251600920166780929,2.5,0.166667,36.166667,1.333333,31.833333
1367234890731773952,1.0,0.333333,34.666667,2.666667,14.000000
1369108855100219395,3.5,0.125000,32.875000,2.875000,11.541667
...,...,...,...,...,...
1301140863742095362,85.0,0.005848,9.385965,1.374269,7.387914
1419760933656739844,549.0,0.000910,8.351228,1.328480,6.340461
1367583722065846278,8.0,0.058824,8.000000,1.235294,6.676471


In [33]:
big_df.to_pickle("bigdf.pkl")

In [35]:
print(os.getcwd())

/workspace/rumor-detection/notebooks
